In [40]:
from openai import OpenAI
from dotenv import load_dotenv

In [41]:
load_dotenv()
client = OpenAI()

#### Step 1 — Extract Text From PDFs

In [42]:
import fitz  # pymupdf

def extract_text_from_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

#### Step 2 — Chunk Text
Chunk size of 500–1000 tokens works best.

In [43]:
def chunk_text(text, chunk_size=1000, overlap=200):
    words = text.split()
    chunks = []
    
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size - overlap)
        
    return chunks

#### Step 3 — Build Embeddings and Store in FAISS

In [44]:
import faiss
import numpy as np

def embed_texts(texts, model="text-embedding-3-large"):
    vectors = []
    for t in texts:
        emb = client.embeddings.create(model=model, input=t)
        vectors.append(emb.data[0].embedding)
    return np.array(vectors).astype("float32")

#### Step 4 — Build the Vector Store From All PDFs

In [45]:
import os
PDF_FOLDER = "books"

all_chunks = []
metadata = []   # to track which chunk came from which PDF

# Extract, chunk, embed
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"):
        path = os.path.join(PDF_FOLDER, filename)
        print("Processing:", filename)

        text = extract_text_from_pdf(path)
        chunks = chunk_text(text, chunk_size=1000, overlap=200)

        all_chunks.extend(chunks)
        metadata.extend([{"source": filename, "chunk": i} for i in range(len(chunks))])

# Embed all chunks
vectors = embed_texts(all_chunks)

# Build FAISS index
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

Processing: BEML Manual MEMU Final given for Printing.pdf
Processing: Reading material EMU-Electrical.pdf


In [46]:
import os
from datetime import datetime

OUTPUT_DIR = "generated_mcqs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [47]:
def save_mcqs_as_txt(mcq_text: str, extension="txt"):
    """
    Saves MCQs to a file inside generated_mcqs folder.
    Default format = .txt
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"mcqs_{timestamp}.{extension}"
    path = os.path.join(OUTPUT_DIR, filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(mcq_text)

    print(f"MCQs saved to: {path}")
    return path

#### Save MCQs as JSON

In [48]:
import json

def save_mcqs_as_json(mcq_text):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"mcqs_{timestamp}.json"
    path = os.path.join(OUTPUT_DIR, filename)

    with open(path, "w", encoding="utf-8") as f:
        json.dump({"mcqs": mcq_text}, f, indent=2)

    print(f"JSON saved to: {path}")
    return path

#### Save MCQs as DOCX

In [49]:
from docx import Document

def save_mcqs_as_docx(mcq_text):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"mcqs_{timestamp}.docx"
    path = os.path.join(OUTPUT_DIR, filename)

    doc = Document()
    doc.add_heading("Generated MCQs", level=1)
    doc.add_paragraph(mcq_text)
    doc.save(path)

    print(f"DOCX saved to: {path}")
    return path

#### Save MCQs as PDF

In [50]:
from reportlab.pdfgen import canvas

def save_mcqs_as_pdf(mcq_text):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"mcqs_{timestamp}.pdf"
    path = os.path.join(OUTPUT_DIR, filename)

    pdf = canvas.Canvas(path)
    textobject = pdf.beginText(40, 800)
    textobject.setFont("Helvetica", 10)

    for line in mcq_text.split("\n"):
        textobject.textLine(line)

    pdf.drawText(textobject)
    pdf.save()

    print(f"PDF saved to: {path}")
    return path

In [51]:
def generate_mcqs_from_all_books(num_questions: int = 100, model="gpt-4.1-mini"):
    SAMPLE_RATE = 10
    sampled_chunks = all_chunks[::SAMPLE_RATE]
    context = "\n\n".join(sampled_chunks)

    prompt = f"""
    You are an expert educator.

    Using ONLY the following context extracted from multiple PDF textbooks,
    strictly generate **{num_questions} high-quality MCQs** that cover a wide range 
    of concepts across ALL topics and chapters indicating the list of topics 
    covered from all books. Mention and the topic name and the questions under the topic clearly.

    Generate:
    1. A list of topics covered  
    2. For each topic:  
        - MCQs (4 options each)  
        - The correct answers clearly stated
    Ensure clarity, structure, and accuracy.

    Requirements:
    - Cover as many different topics as possible.
    - Each MCQ must test a different concept.
    - MCQs must have 4 options: a, b, c, d.
    - Include correct answer and explanation for each question.
    - Do NOT hallucinate or invent anything not in the context.

    ---- CONTEXT START ----
    {context}
    ---- CONTEXT END ----

    Now generate the MCQs.
    """

    response = client.responses.create(
        model=model,
        input=prompt
    )

    mcq_text = response.output_text

    # SAVE INSTEAD OF PRINT
    save_mcqs_as_txt(mcq_text, extension="txt")
    save_mcqs_as_json(mcq_text)
    save_mcqs_as_docx(mcq_text)
    save_mcqs_as_pdf(mcq_text)

    return None

In [52]:
generate_mcqs_from_all_books()

NameError: name 'logging' is not defined